# Teleportación Cuántica

**Objetivo:** implementar el protocolo completo de teleportación y verificar que la fidelidad es 1.

La teleportación transfiere el estado cuántico de un qubit usando un canal clásico (2 bits) y un par entrelazado. **No viola la relatividad**: los 2 bits clásicos limitan la velocidad.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np

# Estado a teleportar: |ψ⟩ = cos(π/6)|0⟩ + sin(π/6)|1⟩
theta = np.pi / 6
psi_target = np.array([np.cos(theta/2), np.sin(theta/2)])
print(f'Estado a teleportar: {psi_target}')

## Circuito Completo

El protocolo usa 3 qubits:
- q0: qubit de Alice a teleportar
- q1: mitad del par Bell de Alice
- q2: mitad del par Bell de Bob

Pasos: (1) crear par Bell, (2) aplicar Bell measurement en q0,q1, (3) correcciones clásicas en q2.

In [ ]:
qc = QuantumCircuit(3, 2)

# Inicializar q0 con el estado |ψ⟩
qc.initialize(psi_target, 0)
qc.barrier()

# Paso 1: Crear par Bell entre q1 y q2
qc.h(1)
qc.cx(1, 2)
qc.barrier()

# Paso 2: Bell measurement de Alice (q0, q1)
qc.cx(0, 1)
qc.h(0)
qc.barrier()
qc.measure(0, 0)
qc.measure(1, 1)
qc.barrier()

# Paso 3: Correcciones condicionales de Bob (q2)
qc.x(2).c_if(1, 1)   # X si bit clásico 1 es 1
qc.z(2).c_if(0, 1)   # Z si bit clásico 0 es 1

print(qc.draw())

In [ ]:
# Verificar fidelidad: ejecutar sin medición para ver el estado final de q2
qc_nomedida = QuantumCircuit(3)
qc_nomedida.initialize(psi_target, 0)
qc_nomedida.h(1)
qc_nomedida.cx(1, 2)
qc_nomedida.cx(0, 1)
qc_nomedida.h(0)

# Traza parcial — estado de q2 tras post-selección |00⟩ en q0,q1
from qiskit.quantum_info import DensityMatrix, partial_trace

rho = DensityMatrix.from_instruction(qc_nomedida)
# Estado de q2 (índice 2)
rho2 = partial_trace(rho, [0, 1])
print('Matriz densidad de q2 (debe ser |ψ⟩⟨ψ|):')
print(np.round(rho2.data, 3))

In [ ]:
# Fidelidad con el estado objetivo
from qiskit.quantum_info import state_fidelity
rho_target = DensityMatrix(psi_target)
fid = state_fidelity(rho2, rho_target)
print(f'Fidelidad: {fid:.6f}  (ideal = 1.0)')

## Canal Ruidoso

En hardware real, el canal de entrelazamiento introduce ruido. Simulemos con depolarizante:

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit import transpile

fidelidades = []
p_vals = np.linspace(0, 0.3, 15)

for p in p_vals:
    noise = NoiseModel()
    noise.add_all_qubit_quantum_error(depolarizing_error(p, 1), ['h', 'initialize'])
    noise.add_all_qubit_quantum_error(depolarizing_error(p, 2), ['cx'])
    
    sim = AerSimulator(noise_model=noise)
    qc_t = transpile(qc_nomedida, sim)
    job = sim.run(qc_t, shots=1)
    
    # Aproximación: reducir fidelidad analíticamente
    fid_approx = (1 - p)**6  # 6 puertas aproximado
    fidelidades.append(fid_approx)

print('p=0.00 → F={:.3f}'.format(fidelidades[0]))
print('p=0.10 → F={:.3f}'.format(fidelidades[7]))
print('p=0.30 → F={:.3f}'.format(fidelidades[-1]))

## Ejercicio

1. ¿Por qué se necesitan exactamente 2 bits clásicos?
2. Teleporta el estado |−⟩ = (|0⟩−|1⟩)/√2 y verifica la fidelidad.
3. ¿Qué pasa si Alice omite la corrección Z?

In [ ]:
# Tu solución
minus = np.array([1, -1]) / np.sqrt(2)
print('Teleportando |−⟩...')
rho_minus = DensityMatrix(minus)
print('Estado objetivo:', rho_minus.data.round(3))

## Próximos pasos

- **Lab completo:** `Cuadernos/laboratorios/02_teleportacion_guiada.ipynb`
- **Módulo:** `Tutorial/01_fundamentos/README.md` (sección entrelazamiento)
- **Siguiente guía:** `05_transformada_de_fourier.ipynb`